# 3D Structure and Chemical Dynamics

So far we've worked with 2D representations of molecules. In this notebook, we'll explore:

1. **3D molecular visualization** using py3Dmol — seeing molecules as they actually exist in space
2. **Chemical kinetics** — simulating how reactions evolve over time using differential equations
3. **PubChem API** — fetching real chemical data from the world's largest chemistry database

This parallels the Birdsong project's "Web Audio" notebook (fetching data from the web) and the Climate project's "Satellite & Reanalysis" notebook (spatial data and dynamics).

## Step 1: Install & Import

In [ ]:
!pip install rdkit-pypi py3Dmol pubchempy -q

from rdkit import Chem
from rdkit.Chem import AllChem, Draw
import py3Dmol
import pubchempy as pcp
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

## Step 2: 3D Molecular Visualization

Molecules aren't flat — they exist in 3D. Bond angles, conformations, and spatial arrangement determine how molecules interact. py3Dmol lets us rotate and inspect molecules interactively.

In [ ]:
def show_3d(smiles, style="stick", width=500, height=400):
    """Generate a 3D conformation and display it interactively."""
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDG())
    AllChem.MMFFOptimizeMolecule(mol)
    mb = Chem.MolToMolBlock(mol)

    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(mb, "mol")
    viewer.setStyle({style: {"colorscheme": "Jmol"}})
    viewer.zoomTo()
    return viewer

# Aspirin in 3D
print("Aspirin (CC(=O)Oc1ccccc1C(=O)O)")
show_3d("CC(=O)Oc1ccccc1C(=O)O").show()

In [ ]:
# Compare flat vs. 3D molecules
molecules_3d = {
    "Benzene (flat, aromatic)": "c1ccccc1",
    "Cyclohexane (chair, 3D)": "C1CCCCC1",
    "Caffeine": "Cn1c(=O)c2c(ncn2C)n(C)c1=O",
    "Glucose": "OC[C@@H](O1)[C@@H](O)[C@H](O)[C@@H](O)[C@@H]1O",
}

for name, smi in molecules_3d.items():
    print(f"\n{name}")
    show_3d(smi).show()

## Step 3: PubChem API — The World's Chemistry Database

[PubChem](https://pubchem.ncbi.nlm.nih.gov/) contains data on over 100 million compounds. We can fetch molecular properties programmatically.

In [ ]:
# Look up a compound by name
compound = pcp.get_compounds("aspirin", "name")[0]

print(f"Name: {compound.iupac_name}")
print(f"Formula: {compound.molecular_formula}")
print(f"MW: {compound.molecular_weight}")
print(f"SMILES: {compound.canonical_smiles}")
print(f"InChI: {compound.inchi}")
print(f"PubChem CID: {compound.cid}")
print(f"XLogP: {compound.xlogp}")

In [ ]:
# Fetch properties for multiple compounds
drug_names = ["aspirin", "ibuprofen", "acetaminophen", "caffeine", "naproxen",
              "penicillin", "metformin", "omeprazole", "atorvastatin", "diazepam"]

print(f"{'Name':20s} {'Formula':15s} {'MW':>8s} {'XLogP':>7s}")
print("-" * 55)

drug_data = []
for name in drug_names:
    try:
        results = pcp.get_compounds(name, "name")
        if results:
            c = results[0]
            print(f"{name:20s} {c.molecular_formula:15s} {c.molecular_weight:>8s} {str(c.xlogp):>7s}")
            drug_data.append({
                "name": name.capitalize(),
                "formula": c.molecular_formula,
                "mw": float(c.molecular_weight),
                "xlogp": float(c.xlogp) if c.xlogp else 0,
                "smiles": c.canonical_smiles,
            })
    except Exception as e:
        print(f"{name:20s} Error: {e}")

In [ ]:
# Visualize the PubChem drug data
if drug_data:
    fig, ax = plt.subplots(figsize=(10, 6))
    names = [d["name"] for d in drug_data]
    mws = [d["mw"] for d in drug_data]
    xlogps = [d["xlogp"] for d in drug_data]

    scatter = ax.scatter(mws, xlogps, s=120, c=range(len(names)),
                         cmap="tab10", edgecolors="black", zorder=3)
    for i, name in enumerate(names):
        ax.annotate(name, (mws[i], xlogps[i]),
                    textcoords="offset points", xytext=(6, 6), fontsize=9)

    ax.axhline(5, color="red", linestyle="--", alpha=0.4)
    ax.axvline(500, color="red", linestyle="--", alpha=0.4)
    ax.set_xlabel("Molecular Weight (g/mol)")
    ax.set_ylabel("XLogP")
    ax.set_title("PubChem Drugs in Chemical Space")
    ax.grid(True, alpha=0.3)
    plt.show()

## Step 4: Chemical Kinetics — Reactions Over Time

Chemistry isn't static. Reactions evolve over time following rate laws described by **ordinary differential equations (ODEs)**.

### First-Order Reaction: A -> B

The simplest reaction: compound A decays into product B at a rate proportional to its concentration.

$$\frac{d[A]}{dt} = -k[A], \qquad \frac{d[B]}{dt} = k[A]$$

In [ ]:
def first_order(y, t, k):
    A, B = y
    dAdt = -k * A
    dBdt = k * A
    return [dAdt, dBdt]

k = 0.5     # rate constant
A0 = 1.0    # initial concentration of A
t = np.linspace(0, 10, 200)

solution = odeint(first_order, [A0, 0], t, args=(k,))

plt.figure(figsize=(8, 5))
plt.plot(t, solution[:, 0], "b-", linewidth=2, label="[A] (reactant)")
plt.plot(t, solution[:, 1], "r-", linewidth=2, label="[B] (product)")
plt.xlabel("Time")
plt.ylabel("Concentration")
plt.title(f"First-Order Reaction: A → B (k = {k})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

half_life = np.log(2) / k
print(f"Half-life: t₁/₂ = ln(2)/k = {half_life:.2f}")

### Reversible Reaction: A ⇌ B

Many reactions are reversible — products can convert back to reactants.

$$\frac{d[A]}{dt} = -k_f[A] + k_r[B]$$

In [ ]:
def reversible(y, t, kf, kr):
    A, B = y
    dAdt = -kf * A + kr * B
    dBdt = kf * A - kr * B
    return [dAdt, dBdt]

kf, kr = 0.5, 0.2  # forward and reverse rate constants
t = np.linspace(0, 15, 300)

solution = odeint(reversible, [1.0, 0.0], t, args=(kf, kr))

plt.figure(figsize=(8, 5))
plt.plot(t, solution[:, 0], "b-", linewidth=2, label="[A]")
plt.plot(t, solution[:, 1], "r-", linewidth=2, label="[B]")
plt.axhline(kr/(kf+kr), color="blue", linestyle="--", alpha=0.4, label=f"[A]_eq = {kr/(kf+kr):.2f}")
plt.axhline(kf/(kf+kr), color="red", linestyle="--", alpha=0.4, label=f"[B]_eq = {kf/(kf+kr):.2f}")
plt.xlabel("Time")
plt.ylabel("Concentration")
plt.title(f"Reversible Reaction: A ⇌ B (kf={kf}, kr={kr})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Keq = kf / kr
print(f"Equilibrium constant K = kf/kr = {Keq:.2f}")
print(f"At equilibrium: [B]/[A] = {Keq:.2f}")

### Enzyme Kinetics: Michaelis-Menten

Enzymes catalyze reactions with a saturation curve described by the Michaelis-Menten equation:

$$v = \frac{V_{max}[S]}{K_m + [S]}$$

In [ ]:
S = np.linspace(0, 20, 200)
Vmax = 10
Km_values = [1, 3, 8]

plt.figure(figsize=(8, 5))
for Km in Km_values:
    v = Vmax * S / (Km + S)
    plt.plot(S, v, linewidth=2, label=f"Km = {Km}")

plt.axhline(Vmax, color="gray", linestyle="--", alpha=0.5, label=f"Vmax = {Vmax}")
plt.axhline(Vmax/2, color="gray", linestyle=":", alpha=0.3)
plt.xlabel("[Substrate]")
plt.ylabel("Reaction Rate (v)")
plt.title("Michaelis-Menten Enzyme Kinetics")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Km = substrate concentration at half-maximal velocity")
print("Lower Km = higher enzyme affinity for the substrate")

## Step 5: Explore Your Own Molecule

Change the SMILES below to visualize any molecule in 3D and look it up on PubChem!

In [ ]:
# ====== CHANGE THIS ======
MY_SMILES = "CC(=O)Oc1ccccc1C(=O)O"   # Try any SMILES string
MY_NAME = "aspirin"                     # Name for PubChem lookup
# =========================

# 3D view
print("3D Structure:")
show_3d(MY_SMILES).show()

# PubChem lookup
try:
    results = pcp.get_compounds(MY_NAME, "name")
    if results:
        c = results[0]
        print(f"\nPubChem Data for {MY_NAME}:")
        print(f"  IUPAC: {c.iupac_name}")
        print(f"  Formula: {c.molecular_formula}")
        print(f"  MW: {c.molecular_weight}")
        print(f"  XLogP: {c.xlogp}")
except Exception as e:
    print(f"PubChem lookup failed: {e}")

## What's Next

In **Notebook 5**, you'll choose your own molecules — from PubChem or your own research — run the full descriptor + autoencoder pipeline, and see where they cluster in chemical space.